In [1]:
import sys

sys.path.append("../")
from agents.trainers.ppo_trainer import PPOTrainer
import gymnasium as gym
import torch
from configs.agents.ppo import PPOConfig
from configs.games.cartpole import CartPoleConfig
from stats.stats import StatTracker

env = gym.make("CartPole-v1", render_mode="rgb_array")
config_dict = {
    "model_name": "ppo_smoke_test",
    # --- New Modular Architecture ---
    "architecture": {
        "activation": "tanh",
        "kernel_initializer": "orthogonal",
    },
    # --- PPO Specific Hyperparameters ---
    "clip_param": 0.2,
    "discount_factor": 0.99,
    "gae_lambda": 0.95,
    "steps_per_epoch": 512,
    "train_policy_iterations": 4,
    "train_value_iterations": 4,
    "target_kl": 0.02,
    "entropy_coefficient": 0.01,
    "num_minibatches": 4,
    "training_steps": 500,
    # --- New Modular Heads ---
    "policy_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },
    },
    "value_head": {
        "neck": {
            "type": "dense",
            "widths": [64],
        },
    },
    # --- Optimization Blocks ---
    "actor_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
    "critic_config": {
        "learning_rate": 2.5e-4,
        "adam_epsilon": 1e-7,
        "clipnorm": 0.5,
    },
    # --- New Action Selector Config ---
    "action_selector": {
        "base": {
            "type": "categorical",  # Base selector type (from registry)
        },
        "decorators": [
            {
                "type": "ppo_injector",  # Decorator to inject log_prob & value
                "kwargs": {},
            }
        ],
    },
}
print("PPO Config")
config = PPOConfig(
    config_dict,
    CartPoleConfig(),
)

trainer = PPOTrainer(
    config,
    env,
    torch.device("cpu"),
    "ppo",
    StatTracker("ppo"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 100
trainer.test_trials = 50

trainer.train()

PPO Config
Using         training_steps                : 500
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 4
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using default minibatch_size                : 64
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 64
Using default n_step                        : 1
Using         discount_factor               : 0.99
Using default per_alpha    

Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'test_score' with subkeys ['avg', 'min', 'max', 'p0']
Initializing stat 'policy_loss' with subkeys None
Initializing stat 'value_loss' with subkeys None
Initializing stat 'policy_entropy' with subkeys None
Initializing stat 'kl_divergence' with subkeys None
Initializing stat 'explained_variance' with subkeys None
Initializing stat 'learner_fps' with subkeys None
Starting PPO training for 500 steps...
Step 0, Avg Score: 24.85, Episodes Finished: 20
Step 10, Avg Score: 24.85, Episodes Finished: 0
Step 10, Avg Score: 48.10, Episodes Finished: 10
Step 20, Avg Score: 48.10, Episodes Finished: 0
Step 20, Avg Score: 167.50, Episodes Finished: 2
Step 30, Avg Score: 167.50, Episodes Finished: 0
Step 30, Avg Score: 127.50, Episodes Finished: 4
Step 40, Avg Score: 127.50, Episodes Finished: 0
Step 40, Avg Score: 212.50, Episodes Finished: 2
Step 50, Avg Score: 212.50, Episodes Finish

In [ ]:
import gymnasium as gym
import sys

sys.path.append("../")
from agents.trainers.rainbow_trainer import RainbowTrainer
from configs.games.cartpole import CartPoleConfig
from configs.agents.rainbow_dqn import RainbowConfig
from stats.stats import StatTracker
import torch

# env = ClipReward(AtariPreprocessing(gym.make("MsPacmanNoFrameskip-v4", render_mode="rgb_array"), terminal_on_life_loss=True), -1, 1) # as recommended by the original paper, should already include max pooling
# env = FrameStack(env, 4)
env = gym.make("CartPole-v1")

config_dict = {
    # --- General Params ---
    "adam_epsilon": 1e-8,
    "learning_rate": 0.001,
    "training_steps": 5000,
    "minibatch_size": 128,
    "transfer_interval": 100,
    "n_step": 3,
    "replay_interval": 1,
    "kernel_initializer": "orthogonal",
    "clipnorm": 10.0,
    "model_name": "rainbow_smoke_test",
    # --- Distributional / C51 Strategy ---
    # Global noisy_sigma for the architecture
    "noisy_sigma": 0.5,
    # --- Backbones ---
    "backbone": {
        "type": "dense",
        "widths": [128],
    },
    # --- Head Configuration (New Nested System) ---
    "head": {
        "output_strategy": {
            "type": "c51",
            "num_atoms": 51,
            # Defaults for v_min/v_max are handled automatically
            # or you can specify them here:
            "v_min": 0,
            "v_max": 500,
        },
        # Hidden layers for Dueling Head (restored to 512 for performance)
        "value_hidden_widths": [],
        "advantage_hidden_widths": [],
        # "noisy_sigma": 0.5 (inherits from global)
    },
    # --- PER (Prioritized Experience Replay) ---
    "per_epsilon": 1e-6,
    "per_alpha": 0.2,
    "per_beta": 0.6,
    "action_selector": {
        "base": {
            "type": "argmax",  # Maps to ArgmaxSelector (Greedy)
            "kwargs": {},
        },
    },
}


game_config = CartPoleConfig()
config = RainbowConfig(config_dict, game_config)
trainer = RainbowTrainer(
    config,
    env,
    torch.device("cpu"),
    "rainbow_refactor",
    StatTracker("rainbow_refactor"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 500

trainer.train()

Using         training_steps                : 5000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using         clipnorm                      : 10.0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 128
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 128
Using         n_step                        : 3
Using default discount_factor               : 0.99
Using         per_alpha         

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'StatTracker'

In [1]:
import sys

sys.path.append("../")
import time
import torch
from agents.random import RandomAgent

# from agents.muzero import MuZeroAgent
from agents.trainers.muzero_trainer import MuZeroTrainer
from configs.agents.muzero import MuZeroConfig
from configs.games.tictactoe import TicTacToeConfig
from agents.tictactoe_expert import TicTacToeBestAgent
from modules.world_models.muzero_world_model import MuzeroWorldModel
from stats.stats import StatTracker

# Ensure we use CPU for fairness/comparibility or GPU if available
device = "cpu"  # or "cuda" if available
print(f"Using device: {device}")

params = {
    # General Training Params
    "num_simulations": 25,
    "training_steps": 10000,
    "transfer_interval": 1,
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "num_workers": 4,
    "search_batch_size": 0,
    # Search & Learning
    "per_alpha": 0.0,
    "per_beta_schedule": {
        "type": "constant",
        "initial": 0.0,
    },
    # If you had temperature scheduling (MuZero typically does), add:
    "temperature_schedule": {
        "type": "stepwise",
        "steps": [5],  # after 5 moves in episode
        "values": [1.0, 0.0],  # start at 1.0, switch to 0.0 after step 5
    },
    "n_step": 10,
    "root_dirichlet_alpha": 0.25,
    "known_bounds": [-1, 1],
    # Action Selector (Required by AgentConfig)
    # Note: MuZeroTrainer currently manually instantiates MCTSDecorator wrapping this,
    # but we define the base strategy here for validation.
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    # Architecture
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24],
        "kernel_sizes": [3],
        "strides": [1],
        "residual_layers": [(24, 3, 1)],
    },
    # Specialized Heads
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        # output_strategy defaults to categorical
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    # Advanced / Stochastic MuZero
    "stochastic": False,
    "gumbel": False,
    "gumbel_m": 16,
    "use_virtual_mean": False,
    "virtual_loss": 3.0,
    "atom_size": 1,
    # Class References
    "world_model_cls": MuzeroWorldModel,
}
game_config = TicTacToeConfig()

print("--- Running MuZero Comaprison of Changes ---")
params_batched = params.copy()
params_batched["search_batch_size"] = 5
params_batched["use_virtual_mean"] = True
params_batched["num_envs_per_worker"] = 4

env_batch = TicTacToeConfig().make_env()
config_batch = MuZeroConfig(config_dict=params_batched, game_config=game_config)
config_batch.search_batch_size = 5  # Explicitly set

trainer = MuZeroTrainer(
    config_batch,
    env_batch,
    torch.device("cpu"),
    name="muzero_search_refactor",
    stats=StatTracker(name="muzero_search_refactor"),
    test_agents=[RandomAgent(), TicTacToeBestAgent()],
)
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 100

start_time = time.time()
trainer.train()
end_time = time.time()
print(f"MuZero Batched Time: {end_time - start_time:.2f}s")

Using device: cpu
--- Running MuZero Comaprison of Changes ---
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 8
Using         replay_buffer_size            : 100000
Using default min_replay_buffer_size        : 8
Using         n_step                        : 10
Using default discount_

KeyboardInterrupt: 

In [1]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from agents.trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 10000,
    "anticipatory_param": 0.1,
    "replay_interval": 2,
    "num_minibatches": 1,
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    # --- NEW RL BACKBONE ---
    "backbone": {"type": "dense", "widths": [128]},
    # Note: value_backbone and advantage_backbone default to Identity (None)
    # which replaces the old empty list [] behavior.
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    # --- SL CONFIG ---
    "sl_learning_rate": 0.01,
    "sl_momentum": 0.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    # --- NEW SL BACKBONE ---
    "sl_backbone": {"type": "dense", "widths": [128]},
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}

config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp_3"))
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 10000
trainer.train()

ModuleNotFoundError: No module named 'agent_configs.dqn.nfsp_config'

In [ ]:
import sys

sys.path.append("../")
import torch
import gymnasium as gym
import numpy as np
from configs.games.game import GameConfig
from configs.agents.ppo import PPOConfig
from agents.trainers.ppo_trainer import PPOTrainer
from stats.stats import StatTracker


class ContinuousActionWrapper(gym.ActionWrapper):
    def action(self, action):
        # Pendulum expects an array-like of shape (1,)
        if np.isscalar(action):
            return np.array([action], dtype=np.float32)
        return np.asarray(action, dtype=np.float32).reshape(self.action_space.shape)


def make_pendulum_env(render_mode=None):
    env = gym.make("Pendulum-v1", render_mode=render_mode)
    return ContinuousActionWrapper(env)


class ContinuousControlGameConfig(GameConfig):
    def __init__(self):
        super().__init__(
            num_actions=None,
            max_score=0.0,
            min_score=-2000.0,
            is_discrete=False,
            is_image=False,
            is_deterministic=False,
            has_legal_moves=False,
            perfect_information=True,
            multi_agent=False,
            num_players=1,
            make_env=make_pendulum_env,
        )


game_config = ContinuousControlGameConfig()
config = PPOConfig(
    {
        "model_name": "ppo_continuous_smoke_notebook",
        "multi_process": False,
        "num_workers": 1,
        "training_steps": 100000,
        "steps_per_epoch": 1024,
        "train_policy_iterations": 4,
        "train_value_iterations": 4,
        "num_minibatches": 4,
        "clip_param": 0.2,
        "discount_factor": 0.99,
        "gae_lambda": 0.95,
        "target_kl": 0.02,
        "entropy_coefficient": 0.1,
        "architecture": {
            "activation": "tanh",
            "kernel_initializer": "orthogonal",
        },
        "policy_head": {
            "neck": {
                "type": "dense",
                "widths": [32],
            },
            "output_strategy": {
                "type": "gaussian",
                "action_dim": 1,
                "min_log_std": -5.0,
                "max_log_std": -2.0,
            },
        },
        "value_head": {
            "neck": {
                "type": "dense",
                "widths": [32],
            },
        },
        "actor_config": {
            "learning_rate": 1e-3,
            "adam_epsilon": 1e-7,
            "clipnorm": 1.0,
        },
        "critic_config": {
            "learning_rate": 1e-3,
            "adam_epsilon": 1e-7,
            "clipnorm": 1.0,
        },
        "action_selector": {
            "base": {
                "type": "categorical",
            },
            "decorators": [
                {
                    "type": "ppo_injector",
                    "kwargs": {},
                }
            ],
        },
    },
    game_config,
)

trainer = PPOTrainer(
    config,
    game_config.make_env(),
    torch.device("cpu"),
    stats=StatTracker("ppo_continuous_notebook"),
)

# Avoid writing checkpoints during notebook smoke tests.
trainer.checkpoint_interval = 1000
trainer.test_interval = 1000
trainer.test_trials = 100

trainer.train()

Using         model_name                    : ppo_continuous_smoke_notebook
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using         num_minibatches               : 4
Using default training_iterations           : 1
Using default lr_schedule_type              : none
Using default lr_schedule_steps             : []
Using default lr_schedule_values            : []
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using default minibatch_size                : 64
Using default replay_buffer_size            : 5000
Using default min_replay_buffer_size        : 64
Using default n_step                        : 

Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'test_score' with subkeys ['avg', 'min', 'max', 'p0']
Initializing stat 'policy_loss' with subkeys None
Initializing stat 'value_loss' with subkeys None
Initializing stat 'policy_entropy' with subkeys None
Initializing stat 'kl_divergence' with subkeys None
Initializing stat 'explained_variance' with subkeys None
Initializing stat 'learner_fps' with subkeys None
Starting PPO training for 10000 steps...
Step 0, Avg Score: -1235.64, Episodes Finished: 5
Step 10, Avg Score: -1235.64, Episodes Finished: 0
Step 10, Avg Score: -1367.49, Episodes Finished: 5
Step 20, Avg Score: -1367.49, Episodes Finished: 0
Step 20, Avg Score: -1272.61, Episodes Finished: 5
Step 30, Avg Score: -1272.61, Episodes Finished: 0
Step 30, Avg Score: -962.99, Episodes Finished: 5
Step 40, Avg Score: -962.99, Episodes Finished: 0
Step 40, Avg Score: -1059.61, Episodes Finished: 5
Step 50, Avg Score: -10